# Time Correction Visualization Pipeline

This notebook provides a step-by-step interactive visualization for evaluating **Time Correction** results.
It compares **Raw** data (Before Time Correction) with **Time-Corrected** data (After Time Correction).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os

# Add the project root to sys.path so we can import CyTof_BatchCorrection
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added to sys.path: {project_root}")


#### This is for the visualization of the Time-corrected results

In [ ]:
import json
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from collections import defaultdict
import collections.abc
collections.MutableMapping = collections.abc.MutableMapping
import scanpy as sc
import seaborn as sns
import anndata as ad
from matplotlib.lines import Line2D
%matplotlib inline
from src.utils import *
from src.time_correction_visualization import *
from src.batch_correction_visualization import *

#### Main pipline for the Time Correction Visualization

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────────
study_config_path = "../results/cohort_1/study_config_cohort1_D0.json"

# Set BOTH to process one panel/dataset at a time (recommended — keeps memory low).
# Leave None to auto-select: dataset → study config value, panel → first found.
target_dataset_override = None     # e.g. "D0"
target_panel_override   = "Panel1" # e.g. "Panel1" or "Panel2"
# ─────────────────────────────────────────────────────────────────────────────

with open(study_config_path) as f:
    study_config = json.load(f)

config = get_config(config_dict=study_config["cytof_config"])
ctx    = CyTOFContext(config)

target_dataset = target_dataset_override or study_config["dataset"]
anchor_name    = ctx.anchors.get(target_dataset)

with open(study_config["outputs"]["markers_path"]) as f:
    markers = json.load(f)

raw_path          = study_config["outputs"]["matrix_path"]
intermediate_path = study_config["outputs"]["matrix_intermediate_path"]

df_raw = pd.read_csv(raw_path)
df_a   = pd.read_csv(intermediate_path)

# Filter to target dataset
if target_dataset and "dataset" in df_raw.columns:
    df_raw = df_raw[df_raw["dataset"] == target_dataset].copy()
    df_a   = df_a[df_a["dataset"] == target_dataset].copy()

# Select the panel to analyse — default to first available
available_panels = sorted(df_raw["panel"].unique()) if "panel" in df_raw.columns else [study_config.get("panel", "Panel1")]
target_panel = target_panel_override or available_panels[0]

# Keep ONLY the target panel in memory
df_raw = df_raw[df_raw["panel"] == target_panel].copy()
df_a   = df_a[df_a["panel"] == target_panel].copy()

# Apply arcsinh to raw data
numeric_markers = [m for m in markers if m in df_raw.columns]
df_b = arcsinh(df_raw, numeric_markers)
del df_raw

print(f"Dataset  : {target_dataset}")
print(f"Panel    : {target_panel}")
print(f"Anchor   : {anchor_name}")
print(f"Before (arcsinh raw)            : {len(df_b)} cells")
print(f"After  (arcsinh time-corrected) : {len(df_a)} cells")
print(f"Markers ({len(markers)}): {markers[:5]}...")

Dataset  : D0
Panel    : Panel1
Anchor   : hBM1012_ANCHOR
Before (arcsinh raw)            : 2630794 cells
After  (arcsinh time-corrected) : 2630794 cells
Markers (76): ['CD45', 'DNA', 'CD44', 'CD61', 'CD38']...


In [4]:
# --- GENERATE & LOAD STUDY-SPECIFIC COLORS ---
_results_dir     = os.path.dirname(study_config["outputs"]["matrix_corrected_path"])
_study_name      = study_config.get("study_name", "study")
colors_save_path = os.path.join(_results_dir, f"colors_dict_{_study_name}.pkl")

if df_a is not None and not os.path.exists(colors_save_path):
    print("\n===== GENERATING GLOBAL COLORS FROM GATED DATA =====")
    all_cell_types = set()
    tc_dir = os.path.join(ctx.output_dir, "time_corrected")
    if os.path.exists(tc_dir):
        for f in glob(os.path.join(tc_dir, "*.csv")):
            try:
                df_temp = pd.read_csv(f, usecols=["cell_type"])
                all_cell_types.update(df_temp["cell_type"].dropna().unique())
            except: pass
    cols = ["celltypes", "panel", "plate", "sample_name", "dataset"]
    temp_df = df_a[cols].drop_duplicates().copy()
    for col in cols:
        if col not in temp_df.columns:
            temp_df[col] = np.nan
    temp_df["cell_type"] = pd.Series(list(all_cell_types))
    generate_and_save_colors(df=temp_df, save_path=colors_save_path)
elif os.path.exists(colors_save_path):
    print(f"[INFO] Using existing colors from {colors_save_path}")

global_colors = {}
if os.path.exists(colors_save_path):
    with open(colors_save_path, "rb") as f_in:
        global_colors = pickle.load(f_in)
    print(f"[INFO] Global colors loaded ({len(global_colors)} keys)")

[INFO] Using existing colors from ../results/cohort_1/colors_dict_cohort_1_D0.pkl
[INFO] Global colors loaded (4 keys)


In [ ]:
import gc

factors = [c for c in ["plate", "group", "celltypes", "Time"] if c in df_b.columns]

# df_b and df_a are already filtered to target_panel — no loop needed
panel_markers = [
    m for m in markers
    if m in df_b.columns
    and m in ctx.true_markers
    and df_b[m].notna().any()
]
print(f"Panel    : {target_panel}  |  Dataset: {target_dataset}")
print(f"Markers  : {len(panel_markers)} | Before : {len(df_b)} | After : {len(df_a)}")

# Per-panel colors
current_colors = {}
if global_colors and os.path.exists(colors_save_path):
    current_colors = load_and_subset_colors(df_a, colors_save_path)

# ── 1. PCA Variance Analysis ──────────────────────────────────────────
print(f"\n[ANALYZE] Running variation_pca for {target_panel}...")
var_b = variation_pca(df_b, markers=panel_markers, group_col="condition",
                      info_cols=factors, panel=target_panel)
var_a = variation_pca(df_a, markers=panel_markers, group_col="condition",
                      info_cols=factors, panel=target_panel)

print("\nVariance Results (Before Time Correction):")
print(var_b)
print("\nVariance Results (After Time Correction):")
print(var_a)

# ── 2. UMAP ───────────────────────────────────────────────────────────
print(f"\n[ANALYZE] Generating UMAP for {target_panel}...")
try:
    adata = umap_by_celltype(df_b, df_a, panel=target_panel,
                             markers=panel_markers, celltype_col="celltypes",
                             subsample=50000)
    plt.show()
except Exception as _e:
    print(f"  [WARN] UMAP skipped: {_e}")

# ── 3. Signal Distributions ───────────────────────────────────────────
print(f"\n[ANALYZE] Running signal distribution plots for {target_panel}...")
plot_signal_distributions_joint(df_before=df_b, df_after=df_a,
                                panel=target_panel, markers=panel_markers,
                                save_dir=None, subsample=50000)
plt.show()

# ── 4. KL Divergence + MWU ────────────────────────────────────────────
print(f"\n[ANALYZE] Calculating KL divergence and MWU for {target_panel}...")
kd_df  = calculate_kd(df_before=df_b, df_after=df_a, panel=target_panel, ctx=ctx)
mwu_df = calculate_mwu(df_before=df_b, df_after=df_a, panel=target_panel, ctx=ctx)
plot_kd_distribution(kd_df, save_path=None)
plt.show()
plot_kd_with_pvalue(kd_df, mwu_df, save_path=None)
plt.show()

# ── 5. Marker-level Variance Boxplot ──────────────────────────────────
print(f"\n[ANALYZE] Generating Marker-level variance boxplots for {target_panel}...")
base_panel_dir = os.path.join(ctx.output_dir, target_dataset, target_panel)
path_Variance  = os.path.join(base_panel_dir, "Time_Variance")
os.makedirs(path_Variance, exist_ok=True)

mvar_b = variation_marker(df_b, markers=panel_markers, group_col=None,
                          info_cols=factors, panel=target_panel)
mvar_a = variation_marker(df_a, markers=panel_markers, group_col=None,
                          info_cols=factors, panel=target_panel)

mvar_b.to_csv(os.path.join(path_Variance,
    f"variance_marker_raw_before_{target_dataset}_{target_panel}_Time.csv"), index=False)
mvar_a.to_csv(os.path.join(path_Variance,
    f"variance_marker_raw_after_{target_dataset}_{target_panel}_Time.csv"), index=False)

plot_variance_marker_boxplot(
    var_marker_raw=mvar_b,
    var_marker_corr=mvar_a,
    save_path=os.path.join(path_Variance,
        f"variance_marker_boxplot_{target_dataset}_{target_panel}_Time.png"),
    use_log=True
)
plt.show()

try:
    del adata
except NameError:
    pass
gc.collect()
print(f"\n[DONE] Finished panel: {target_panel}")

In [ ]:
import gc

# ── Cross-Panel Variance Analysis ─────────────────────────────────────────────
# Re-reads BOTH panels from the study config paths.
# (df_b / df_a above hold only one panel to keep memory usage low.)

# ── USER INPUT ────────────────────────────────────────────────────────────────
p1_name_override = None   # e.g. "Panel1"
p2_name_override = None   # e.g. "Panel2"
# ─────────────────────────────────────────────────────────────────────────────

try:
    del adata
except NameError:
    pass
try:
    del df_b, df_a
except NameError:
    pass
gc.collect()

raw_path          = study_config["outputs"]["matrix_path"]
intermediate_path = study_config["outputs"]["matrix_intermediate_path"]

print("[INFO] Re-reading both panels for cross-panel analysis...")
_df_raw  = pd.read_csv(raw_path)
df_all_a = pd.read_csv(intermediate_path)

if target_dataset and "dataset" in _df_raw.columns:
    _df_raw  = _df_raw[_df_raw["dataset"] == target_dataset].copy()
    df_all_a = df_all_a[df_all_a["dataset"] == target_dataset].copy()

_numeric = [m for m in markers if m in _df_raw.columns]
df_all_b = arcsinh(_df_raw, _numeric)
del _df_raw

available_panels = sorted(df_all_b["panel"].unique()) if "panel" in df_all_b.columns else []
print(f"[INFO] Available panels: {available_panels}")

if len(available_panels) < 2:
    print("[SKIP] Cross-panel analysis requires at least 2 panels. Skipping.")
else:
    p1_name = p1_name_override or available_panels[0]
    p2_name = p2_name_override or available_panels[1]

    _factors     = [c for c in ["plate", "group", "celltypes", "Time"] if c in df_all_b.columns]
    _results_dir = os.path.dirname(study_config["outputs"]["matrix_corrected_path"])
    _vis_dir     = os.path.join(_results_dir, "cross_panel_visualization_time")
    os.makedirs(_vis_dir, exist_ok=True)

    try:
        df_p1_b = df_all_b[df_all_b["panel"] == p1_name].copy()
        df_p1_a = df_all_a[df_all_a["panel"] == p1_name].copy()
        markers_p1 = [m for m in markers if m in df_p1_b.columns and df_p1_b[m].notna().any()]

        df_p2_b = df_all_b[df_all_b["panel"] == p2_name].copy()
        df_p2_a = df_all_a[df_all_a["panel"] == p2_name].copy()
        markers_p2 = [m for m in markers if m in df_p2_b.columns and df_p2_b[m].notna().any()]

        print(f"\n[ANALYZE] PCA for {p1_name} ({len(markers_p1)} markers)...")
        var_p1_b = variation_pca(df_p1_b, markers=markers_p1, group_col="condition", info_cols=_factors, panel=p1_name)
        var_p1_a = variation_pca(df_p1_a, markers=markers_p1, group_col="condition", info_cols=_factors, panel=p1_name)

        print(f"\n[ANALYZE] PCA for {p2_name} ({len(markers_p2)} markers)...")
        var_p2_b = variation_pca(df_p2_b, markers=markers_p2, group_col="condition", info_cols=_factors, panel=p2_name)
        var_p2_a = variation_pca(df_p2_a, markers=markers_p2, group_col="condition", info_cols=_factors, panel=p2_name)

        print(f"\n[POST-PROCESS] Cross-panel variance: {p1_name} vs {p2_name}")
        plot_variance_bar(var_p1_b, var_p1_a, var_p2_b, var_p2_a,
                          save_path=os.path.join(_vis_dir, "global_variance_bar_time.png"))
        plt.show()

        del df_p1_b, df_p1_a, df_p2_b, df_p2_a
    except Exception as _e:
        import traceback
        print(f"[ERROR] Cross-panel (time) visualization failed: {_e}")
        traceback.print_exc()

    del df_all_b, df_all_a
    gc.collect()